# 01 — Data Preprocessing & Feature Engineering

This notebook takes raw delivery CSVs and produces two analysis-ready datasets:

- **`panel_weekly.parquet`** — Monday-aligned weekly demand panel for every (nest, facility, product), zero-filled, with demand classification and train/valid/test splits.
- **`unified_features.parquet`** — 40-feature supervised table for modellable pairs, ready for the forecasting model.

**Data input**: Upload delivery CSVs to a Google Drive folder and set the path below. The notebook auto-discovers all files with the required columns — no code changes needed to add new nests or datasets.

## Step 0 — Imports and Configuration

In [2]:
from pathlib import Path
import time
import warnings
import numpy as np
import pandas as pd
from google.colab import drive, files
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings('ignore')
pd.set_option('display.width', 140)
pd.set_option('display.max_columns', 60)

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# ── Set your Google Drive data folder path here ──
DATA_DIR = Path('/content/drive/MyDrive/data')

PAIR_COLS = ['NEST_NAME', 'HEALTH_FACILITY_NAME', 'PRODUCT_NAME']
CORE_COLS = ['DATE', 'NEST_NAME', 'SHIPMENT_PRIORITY', 'HEALTH_FACILITY_NAME',
             'UNITS_DELIVERED', 'PRODUCT_NAME']

PRIORITY_SCOPE = ['custom window', 'resupply', 'scheduled', 'fast']

# Syntetos-Boylan ADI-CV² thresholds
ADI_CUTOFF = 1.32
CV2_CUTOFF = 0.49

# Tier rules
MIN_NONZERO_WEEKS = 5
MAX_ADI           = 15.0
RECENCY_WEEKS     = 26

# Per-nest calendar split
N_TEST_WEEKS  = 5
N_VALID_WEEKS = 5

t0 = time.time()

## Step 1 — Auto-Discover and Load Raw CSVs

We scan `data/` for all CSVs and load those with the six required columns. Files missing any column are skipped — so output files or unrelated data won't break anything. To add a new nest, just drop its CSV here and re-run.

In [4]:
all_csvs = sorted(DATA_DIR.glob('*.csv'))
print(f'Found {len(all_csvs)} CSV file(s) in {DATA_DIR.name}/\n')

frames = []
skipped = []
for p in all_csvs:
    try:
        header_cols = pd.read_csv(p, nrows=0).columns.tolist()
    except Exception as e:
        print(f'  [SKIP] {p.name:<45s}  could not read header: {e}')
        skipped.append(p.name)
        continue

    missing = [c for c in CORE_COLS if c not in header_cols]
    if missing:
        print(f'  [SKIP] {p.name:<45s}  missing columns: {missing}')
        skipped.append(p.name)
        continue

    d = pd.read_csv(p, usecols=CORE_COLS, parse_dates=['DATE'], low_memory=False)
    nests_in_file = d['NEST_NAME'].dropna().unique()
    print(f'  [OK]   {p.name:<45s}  rows={len(d):>9,}   nests={list(nests_in_file)}')
    frames.append(d)

if not frames:
    raise FileNotFoundError(
        f'No valid delivery CSVs found in {DATA_DIR}. '
        f'Each file must contain columns: {CORE_COLS}')

df_raw = pd.concat(frames, ignore_index=True)
print(f'\nLoaded {len(frames)} file(s), concatenated shape: {df_raw.shape}')
if skipped:
    print(f'Skipped {len(skipped)} file(s): {skipped}')
print(f'elapsed={time.time()-t0:.1f}s')

Found 4 CSV file(s) in data/

  [OK]   Copy of RW-1 2025 2026.csv                     rows=  214,573   nests=['RW1 Muhanga']
  [OK]   Copy of RW-2 2025 2026.csv                     rows=  145,531   nests=['RW2 Kayonza']
  [OK]   GH-1 2024 2025 2026 Demand.csv                 rows=   86,300   nests=['GH1 Omenako']
  [OK]   GH-3 2024 2025 2026 Demand.csv                 rows=   98,953   nests=['GH3 Vobsi']

Loaded 4 file(s), concatenated shape: (545357, 6)
elapsed=11.2s


## Step 2 — Clean Data

Drop rows with missing key fields, strip whitespace from strings, and keep only **planned** deliveries — reactive `emergency` shipments are excluded as they represent unpredictable demand.

In [5]:
df = df_raw.dropna(subset=['HEALTH_FACILITY_NAME', 'PRODUCT_NAME',
                           'UNITS_DELIVERED', 'NEST_NAME']).copy()
for c in PAIR_COLS:
    df[c] = df[c].astype(str).str.strip()

before = len(df)
df = df[df['SHIPMENT_PRIORITY'].isin(PRIORITY_SCOPE)].copy()
df['units'] = df['UNITS_DELIVERED'].astype(float)

print(f'After cleaning + priority filter: {before:,} -> {len(df):,} rows')
print('\nPer-nest delivery counts:')
print(df['NEST_NAME'].value_counts())
print(f'\nelapsed={time.time()-t0:.1f}s')

After cleaning + priority filter: 544,168 -> 443,514 rows

Per-nest delivery counts:
NEST_NAME
RW1 Muhanga    152998
RW2 Kayonza    113153
GH3 Vobsi       94734
GH1 Omenako     82629
Name: count, dtype: int64

elapsed=16.3s


## Step 3 — Build Monday-Aligned Weekly Panel

Aggregate daily deliveries into weeks (starting Monday), then zero-fill each (facility, product) pair so it has a continuous time series. Zero-fill is done **per nest** because GH and RW have different date ranges — we don't want to create fake zeros before a nest started operating.

In [6]:
dt = pd.to_datetime(df['DATE'])
df['week_start'] = (dt - pd.to_timedelta(dt.dt.dayofweek, unit='D')).dt.normalize()

weekly = (df.groupby(PAIR_COLS + ['week_start'], as_index=False)['units']
            .sum().rename(columns={'units': 'weekly_units'}))

panels = []
for nest, g in weekly.groupby('NEST_NAME', sort=False):
    weeks = pd.date_range(g['week_start'].min(), g['week_start'].max(), freq='W-MON')
    pairs = g[PAIR_COLS].drop_duplicates()
    full = pairs.merge(pd.DataFrame({'week_start': weeks}), how='cross')
    full = full.merge(g, on=PAIR_COLS + ['week_start'], how='left')
    full['weekly_units'] = full['weekly_units'].fillna(0.0)
    panels.append(full)

panel = pd.concat(panels, ignore_index=True).sort_values(
    PAIR_COLS + ['week_start']).reset_index(drop=True)

print(f'Weekly panel shape: {panel.shape}')
print('\nPairs per nest:')
print(panel.groupby('NEST_NAME')[PAIR_COLS].apply(
    lambda d: d.drop_duplicates().shape[0]))
print('\nWeek range per nest:')
print(panel.groupby('NEST_NAME')['week_start'].agg(['min', 'max', 'nunique']))
print(f'\nelapsed={time.time()-t0:.1f}s')

Weekly panel shape: (4091010, 5)

Pairs per nest:
NEST_NAME
GH1 Omenako     9296
GH3 Vobsi       8667
RW1 Muhanga    18930
RW2 Kayonza    11463
dtype: int64

Week range per nest:
                   min        max  nunique
NEST_NAME                                 
GH1 Omenako 2024-01-01 2026-02-09      111
GH3 Vobsi   2024-01-01 2026-02-09      111
RW1 Muhanga 2024-12-30 2026-04-20       69
RW2 Kayonza 2024-12-30 2026-04-20       69

elapsed=30.1s


## Step 4 — Per-Nest Calendar Split

Split per nest (not globally) since GH and RW have different end dates. Last 5 weeks = test, prior 5 = validation, rest = train.

In [7]:
panel['nest_max_week'] = panel.groupby('NEST_NAME')['week_start'].transform('max')
panel['test_lo']  = panel['nest_max_week'] - pd.Timedelta(weeks=N_TEST_WEEKS - 1)
panel['valid_lo'] = panel['test_lo']       - pd.Timedelta(weeks=N_VALID_WEEKS)

panel['split'] = np.select(
    [panel['week_start'] >= panel['test_lo'],
     panel['week_start'] >= panel['valid_lo']],
    ['test', 'valid'],
    default='train',
)
panel = panel.drop(columns=['nest_max_week', 'test_lo', 'valid_lo'])

print('Split row counts:')
print(panel['split'].value_counts())
print('\nSplit weeks per nest:')
print(panel.groupby(['NEST_NAME', 'split'])['week_start'].agg(['min', 'max', 'nunique']))

Split row counts:
split
train    3607450
valid     241780
test      241780
Name: count, dtype: int64

Split weeks per nest:
                         min        max  nunique
NEST_NAME   split                               
GH1 Omenako test  2026-01-12 2026-02-09        5
            train 2024-01-01 2025-12-01      101
            valid 2025-12-08 2026-01-05        5
GH3 Vobsi   test  2026-01-12 2026-02-09        5
            train 2024-01-01 2025-12-01      101
            valid 2025-12-08 2026-01-05        5
RW1 Muhanga test  2026-03-23 2026-04-20        5
            train 2024-12-30 2026-02-09       59
            valid 2026-02-16 2026-03-16        5
RW2 Kayonza test  2026-03-23 2026-04-20        5
            train 2024-12-30 2026-02-09       59
            valid 2026-02-16 2026-03-16        5


## Step 5 — ADI-CV² Demand Classification & Tier Labelling

Using **training data only** (to prevent leakage), we classify each (facility, product) pair with the Syntetos-Boylan framework:

| | CV² < 0.49 | CV² >= 0.49 |
|---|---|---|
| **ADI < 1.32** (frequent) | Smooth | Erratic |
| **ADI >= 1.32** (infrequent) | Intermittent | Lumpy |

- **ADI** = total weeks / nonzero weeks (how often demand occurs)
- **CV²** = (std / mean)² of nonzero demand (how variable order sizes are)

Pairs with enough history (`nonzero_weeks >= 5`, `ADI <= 15`, active in last 26 weeks) are labelled **modellable** for ML. The rest are **rare** and will use a simple heuristic later.

In [8]:
tr_panel = panel[panel['split'] == 'train']

pair_stats = (tr_panel.groupby(PAIR_COLS, as_index=False)
              .agg(total_weeks=('weekly_units', 'count'),
                   nonzero_weeks=('weekly_units', lambda x: int((x > 0).sum())),
                   mean_nonzero=('weekly_units',
                                 lambda x: float(x[x > 0].mean()) if (x > 0).any() else 0.0),
                   std_nonzero=('weekly_units',
                                lambda x: float(x[x > 0].std()) if (x > 0).sum() > 1 else 0.0)))

pair_stats['adi'] = pair_stats['total_weeks'] / pair_stats['nonzero_weeks'].replace(0, np.nan)
pair_stats['cv2'] = ((pair_stats['std_nonzero'] /
                      pair_stats['mean_nonzero'].replace(0, np.nan)) ** 2).fillna(0)

def classify(r):
    if r['nonzero_weeks'] == 0:
        return 'Zero_only'
    if r['adi'] < ADI_CUTOFF:
        return 'Smooth' if r['cv2'] < CV2_CUTOFF else 'Erratic'
    return 'Intermittent' if r['cv2'] < CV2_CUTOFF else 'Lumpy'

pair_stats['demand_class'] = pair_stats.apply(classify, axis=1)

# Recency check: how many weeks since the pair last had a delivery
last_active = (panel.loc[panel['weekly_units'] > 0]
                    .groupby(PAIR_COLS, as_index=False)['week_start']
                    .max().rename(columns={'week_start': 'last_active_week'}))
nest_max = (panel.groupby('NEST_NAME', as_index=False)['week_start']
                 .max().rename(columns={'week_start': 'nest_max_week'}))
pair_stats = (pair_stats.merge(last_active, on=PAIR_COLS, how='left')
                        .merge(nest_max, on='NEST_NAME', how='left'))
pair_stats['weeks_since_active'] = (
    (pair_stats['nest_max_week'] - pair_stats['last_active_week']).dt.days / 7)

pair_stats['tier'] = np.where(
    (pair_stats['nonzero_weeks'] >= MIN_NONZERO_WEEKS)
    & (pair_stats['adi'] <= MAX_ADI)
    & (pair_stats['weeks_since_active'] <= RECENCY_WEEKS),
    'modellable', 'rare',
)

panel = panel.merge(
    pair_stats[PAIR_COLS + ['demand_class', 'tier', 'adi', 'cv2',
                            'mean_nonzero', 'nonzero_weeks']],
    on=PAIR_COLS, how='left',
)

print('Tier distribution (per pair):')
print(pair_stats['tier'].value_counts())
print('\nDemand class distribution (per pair):')
print(pair_stats['demand_class'].value_counts())
print('\nModellable pairs per nest:')
print(pair_stats.loc[pair_stats['tier'] == 'modellable'].groupby('NEST_NAME').size())
print(f'\nelapsed={time.time()-t0:.1f}s')

Tier distribution (per pair):
tier
rare          40066
modellable     8290
Name: count, dtype: int64

Demand class distribution (per pair):
demand_class
Intermittent    39906
Lumpy            4416
Zero_only        3929
Smooth             95
Erratic            10
Name: count, dtype: int64

Modellable pairs per nest:
NEST_NAME
GH1 Omenako     848
GH3 Vobsi      1906
RW1 Muhanga    3554
RW2 Kayonza    1982
dtype: int64

elapsed=78.7s


## Step 6 — Save the Weekly Panel

Save the full panel (including zero-demand weeks) — Notebook 02 needs it for feature recomputation and the rare-pair heuristic.

In [9]:
panel_out = panel[PAIR_COLS + ['week_start', 'weekly_units', 'demand_class',
                               'tier', 'split', 'mean_nonzero', 'nonzero_weeks']]
panel_out.to_parquet(DATA_DIR / 'panel_weekly.parquet', index=False)

print(f'Saved panel_weekly.parquet')
print(f'  shape: {panel_out.shape}')
print(f'  size:  {(DATA_DIR / "panel_weekly.parquet").stat().st_size / 1e6:.1f} MB')
print(f'  elapsed={time.time()-t0:.1f}s')

Saved panel_weekly.parquet
  shape: (4091010, 10)
  size:  1.1 MB
  elapsed=87.0s


## Step 7 — Feature Engineering (Modellable Pairs Only)

From here we work only with modellable pairs. We build 40 features, all strictly **past-only** — the current week's demand is never visible to its own features.

| Category | Count | What it captures |
|----------|-------|-----------------|
| Calendar | 6 | Seasonality, week-of-year, rainy season |
| Lags (1,2,3,4,8,12) | 6 | Recent demand history |
| Rolling (4/8/13-week windows) | 12 | Short- and medium-term trends, volatility |
| Intermittent signals | 4 | Time since last order, nonzero rate |
| Expanding history | 6 | Long-run facility / product / pair averages |
| Pair stats + encodings | 6 | ADI, CV², encoded identifiers |

In [10]:
feat = panel[panel['tier'] == 'modellable'].copy()
feat = feat.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
print('Modellable feature frame starting shape:', feat.shape)

Modellable feature frame starting shape: (687678, 12)


### 7a. Calendar Features

In [11]:
feat['week_of_year'] = feat['week_start'].dt.isocalendar().week.astype(int)
feat['month']        = feat['week_start'].dt.month
feat['quarter']      = feat['week_start'].dt.quarter
feat['week_sin']     = np.sin(2 * np.pi * feat['week_of_year'] / 52)
feat['week_cos']     = np.cos(2 * np.pi * feat['week_of_year'] / 52)
feat['rainy_season'] = feat['month'].between(4, 10).astype(int)

### 7b. Lag Features (1, 2, 3, 4, 8, 12 Weeks)

In [12]:
g_units = feat.groupby(PAIR_COLS, sort=False)['weekly_units']
for k in [1, 2, 3, 4, 8, 12]:
    feat[f'lag_{k}'] = g_units.shift(k)
print(f'Lags done. elapsed={time.time()-t0:.1f}s')

Lags done. elapsed=119.5s


### 7c. Rolling Features (4, 8, 13-Week Windows)

Computed on the **shifted-by-1** series so the current week is excluded (no target leakage).

In [13]:
shifted_units = feat.groupby(PAIR_COLS, sort=False)['weekly_units'].shift(1)
shifted_occ   = (shifted_units > 0).astype(float)
keys = [feat[c] for c in PAIR_COLS]

for w in [4, 8, 13]:
    feat[f'roll_mean_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).mean())
    feat[f'roll_std_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).std().fillna(0))
    feat[f'roll_max_{w}'] = shifted_units.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).max())
    feat[f'roll_nz_rate_{w}'] = shifted_occ.groupby(keys).transform(
        lambda s: s.rolling(w, min_periods=1).mean())

print(f'Rolling done. elapsed={time.time()-t0:.1f}s')

Rolling done. elapsed=151.2s


### 7d. Intermittent-Demand Features

Per-pair walk through the time series to compute: weeks since last demand, last nonzero order size, historical nonzero rate, and cumulative active weeks.

In [14]:
ws_all, lnz_all, hnz_all, ca_all = [], [], [], []
for _, g in feat.groupby(PAIR_COLS, sort=False):
    units = g['weekly_units'].to_numpy()
    n = len(units)
    ws  = np.full(n, np.nan)
    lnz = np.full(n, np.nan)
    hnz = np.full(n, np.nan)
    ca  = np.zeros(n)
    last_idx, last_val, cum = -1, np.nan, 0
    for i in range(n):
        if i > 0:
            hnz[i] = cum / i
            ca[i]  = cum
            if last_idx >= 0:
                ws[i]  = i - last_idx
                lnz[i] = last_val
        if units[i] > 0:
            last_idx, last_val, cum = i, units[i], cum + 1
    ws_all.extend(ws)
    lnz_all.extend(lnz)
    hnz_all.extend(hnz)
    ca_all.extend(ca)

feat['weeks_since_last_demand'] = ws_all
feat['last_nonzero_units']      = lnz_all
feat['historical_nonzero_rate'] = hnz_all
feat['cumulative_active_weeks'] = ca_all
print(f'Intermittent features done. elapsed={time.time()-t0:.1f}s')

Intermittent features done. elapsed=160.5s


### 7e. Expanding History (Facility / Product / Pair Level)

Long-run averages and activity rates at three levels of aggregation. All use `.shift(1).expanding().mean()` so the current week is excluded.

In [15]:
# Facility-level history
fac = (feat.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], as_index=False)
            ['weekly_units'].sum().rename(columns={'weekly_units': 'fac_total'})
            .sort_values(['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start']))
fac['fac_nz'] = (fac['fac_total'] > 0).astype(int)
gf = fac.groupby(['NEST_NAME', 'HEALTH_FACILITY_NAME'])
fac['facility_hist_mean']    = gf['fac_total'].transform(
    lambda s: s.shift(1).expanding().mean())
fac['facility_hist_nz_rate'] = gf['fac_nz'].transform(
    lambda s: s.shift(1).expanding().mean())
feat = feat.merge(fac[['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start',
                       'facility_hist_mean', 'facility_hist_nz_rate']],
                  on=['NEST_NAME', 'HEALTH_FACILITY_NAME', 'week_start'], how='left')

# Product-level history (within nest)
prod = (feat.groupby(['NEST_NAME', 'PRODUCT_NAME', 'week_start'], as_index=False)
             ['weekly_units'].sum().rename(columns={'weekly_units': 'prod_total'})
             .sort_values(['NEST_NAME', 'PRODUCT_NAME', 'week_start']))
prod['prod_nz'] = (prod['prod_total'] > 0).astype(int)
gp = prod.groupby(['NEST_NAME', 'PRODUCT_NAME'])
prod['product_hist_mean']    = gp['prod_total'].transform(
    lambda s: s.shift(1).expanding().mean())
prod['product_hist_nz_rate'] = gp['prod_nz'].transform(
    lambda s: s.shift(1).expanding().mean())
feat = feat.merge(prod[['NEST_NAME', 'PRODUCT_NAME', 'week_start',
                        'product_hist_mean', 'product_hist_nz_rate']],
                  on=['NEST_NAME', 'PRODUCT_NAME', 'week_start'], how='left')

# Pair-level expanding history
feat = feat.sort_values(PAIR_COLS + ['week_start']).reset_index(drop=True)
g_pair = feat.groupby(PAIR_COLS, sort=False)['weekly_units']
feat['fp_hist_mean'] = g_pair.transform(
    lambda s: s.shift(1).expanding().mean())
feat['fp_nz_tmp'] = (feat['weekly_units'] > 0).astype(int)
feat['fp_hist_nz_rate'] = feat.groupby(PAIR_COLS, sort=False)['fp_nz_tmp'].transform(
    lambda s: s.shift(1).expanding().mean())
feat = feat.drop(columns=['fp_nz_tmp'])

print(f'Expanding history done. elapsed={time.time()-t0:.1f}s')

Expanding history done. elapsed=175.6s


### 7f. Label Encoding

In [16]:
for col in ['HEALTH_FACILITY_NAME', 'PRODUCT_NAME']:
    feat[col + '_enc'] = LabelEncoder().fit_transform(feat[col].astype(str))

print(f'Encodings done. elapsed={time.time()-t0:.1f}s')
print(f'\nUnified feature frame shape: {feat.shape}')
print('\nPer-nest rows:')
print(feat['NEST_NAME'].value_counts(dropna=False))

Encodings done. elapsed=183.2s

Unified feature frame shape: (687678, 48)

Per-nest rows:
NEST_NAME
RW1 Muhanga    245226
GH3 Vobsi      211566
RW2 Kayonza    136758
GH1 Omenako     94128
Name: count, dtype: int64


## Step 8 — Save the Unified Feature Table

This is the direct input for Notebook 02's forecasting model.

In [17]:
FEATURE_COLS = [
    # Calendar
    'week_of_year', 'month', 'quarter', 'week_sin', 'week_cos', 'rainy_season',
    # Lags
    'lag_1', 'lag_2', 'lag_3', 'lag_4', 'lag_8', 'lag_12',
    # Rolling windows
    'roll_mean_4', 'roll_std_4', 'roll_max_4', 'roll_nz_rate_4',
    'roll_mean_8', 'roll_std_8', 'roll_max_8', 'roll_nz_rate_8',
    'roll_mean_13', 'roll_std_13', 'roll_max_13', 'roll_nz_rate_13',
    # Intermittent
    'weeks_since_last_demand', 'last_nonzero_units',
    'historical_nonzero_rate', 'cumulative_active_weeks',
    # Expanding history
    'facility_hist_mean', 'facility_hist_nz_rate',
    'product_hist_mean',  'product_hist_nz_rate',
    'fp_hist_mean',       'fp_hist_nz_rate',
    # Pair-level numerics
    'adi', 'cv2', 'mean_nonzero', 'nonzero_weeks',
    # Encodings
    'HEALTH_FACILITY_NAME_enc', 'PRODUCT_NAME_enc',
]

KEEP_COLS = (PAIR_COLS
             + ['week_start', 'weekly_units', 'demand_class', 'tier', 'split']
             + FEATURE_COLS)

unified = feat[KEEP_COLS].copy()
unified.to_parquet(DATA_DIR / 'unified_features.parquet', index=False)

print(f'Saved unified_features.parquet')
print(f'  shape: {unified.shape}')
print(f'  size:  {(DATA_DIR / "unified_features.parquet").stat().st_size / 1e6:.1f} MB')
print(f'\nFeature columns ({len(FEATURE_COLS)}):')
for c in FEATURE_COLS:
    print(f'  - {c}')
print(f'\nTOTAL elapsed: {time.time()-t0:.1f}s')

Saved unified_features.parquet
  shape: (687678, 48)
  size:  11.5 MB

Feature columns (40):
  - week_of_year
  - month
  - quarter
  - week_sin
  - week_cos
  - rainy_season
  - lag_1
  - lag_2
  - lag_3
  - lag_4
  - lag_8
  - lag_12
  - roll_mean_4
  - roll_std_4
  - roll_max_4
  - roll_nz_rate_4
  - roll_mean_8
  - roll_std_8
  - roll_max_8
  - roll_nz_rate_8
  - roll_mean_13
  - roll_std_13
  - roll_max_13
  - roll_nz_rate_13
  - weeks_since_last_demand
  - last_nonzero_units
  - historical_nonzero_rate
  - cumulative_active_weeks
  - facility_hist_mean
  - facility_hist_nz_rate
  - product_hist_mean
  - product_hist_nz_rate
  - fp_hist_mean
  - fp_hist_nz_rate
  - adi
  - cv2
  - mean_nonzero
  - nonzero_weeks
  - HEALTH_FACILITY_NAME_enc
  - PRODUCT_NAME_enc

TOTAL elapsed: 187.9s
